# 05 — MCP 集成

**来源:** [LangChain Docs — MCP](https://docs.langchain.com/mcp)

Model Context Protocol（MCP）是一种开放协议，允许你将 LangChain Deep Agents 的文档连接到 Claude Desktop、VSCode 等任何支持 MCP 的客户端，实现实时的文档查询。

本笔记涵盖：
- MCP 简介与架构
- 配置 MCP 客户端连接 LangChain 文档
- 使用 MCP 工具查询 Deep Agents 文档
- 将 MCP 集成到 Deep Agent 中
- 实践：用 Python 调用 MCP 服务器

## 1. 什么是 MCP？

Model Context Protocol（模型上下文协议）是一种开放标准，定义了大模型应用与外部数据源和工具之间的通信协议。

| 概念 | 说明 |
|------|------|
| **MCP Server** | 提供数据或功能的服务器（如文档查询、文件系统、数据库） |
| **MCP Client** | 连接到 MCP Server 的客户端（Claude Desktop、VSCode、自定义应用） |
| **Tools** | Server 暴露的工具函数，Client 可调用 |
| **Resources** | Server 暴露的静态或动态数据资源 |
| **Prompts** | Server 提供的预设 Prompt 模板 |

LangChain Docs 通过 MCP 服务器在 `https://docs.langchain.com/mcp` 提供文档查询服务。

## 2. 配置 Claude Desktop

在 Claude Desktop 中，通过 `claude_desktop_config.json` 配置 MCP 服务器：

```json
{
  "mcpServers": {
    "langchain-docs": {
      "type": "url",
      "url": "https://docs.langchain.com/mcp"
    }
  }
}
```

配置完成后，你可以在对话中直接询问 "LangChain Deep Agents 的中文文档" 等，Claude 会自动调用 MCP 查询。

## 3. 配置 VSCode

在 VSCode 中，通过设置扩展配置来连接 MCP 服务器。

编辑 `settings.json`：

```json
{
  "cline.mcpServers": {
    "langchain-docs": {
      "type": "url",
      "url": "https://docs.langchain.com/mcp"
    }
  }
}
```

或根据使用的扩展（如 Continue、Cline）调整配置格式。

## 4. 使用 Python 客户端调用 MCP

以下示例展示如何通过 HTTP POST 调用 LangChain Docs MCP 服务器，查询文档内容。

In [ ]:
import json
import requests

MCP_URL = "https://docs.langchain.com/mcp"

def call_mcp_tool(name: str, arguments: dict) -> dict:
    """调用 MCP Server 上的工具"""
    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": {
            "name": name,
            "arguments": arguments,
        },
    }
    response = requests.post(
        MCP_URL,
        json=payload,
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json, text/event-stream",
        },
    )
    return response.json()

# 列出可用的 MCP 工具
tools_payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
}
try:
    tools_resp = requests.post(
        MCP_URL,
        json=tools_payload,
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json, text/event-stream",
        },
        timeout=10,
    )
    print("MCP 工具列表（需服务器支持 tools/list 方法）：")
    print(tools_resp.text[:500])
except Exception as e:
    print(f"无法获取工具列表（某些 MCP Server 不支持 listings）：{e}")

MCP 工具列表（需服务器支持 tools/list 方法）：
{"jsonrpc":"2.0","id":1,"error":{"code":-32601,"message":"Method not found"}}


### 4.1 查询文档内容

LangChain Docs MCP 服务器暴露了 `query_docs_filesystem_docs_by_lang_chain` 工具，可在文档文件系统上执行命令。

In [ ]:
import requests

MCP_URL = "https://docs.langchain.com/mcp"

def query_docs(command: str) -> str:
    """通过 MCP 查询 LangChain 文档"""
    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": {
            "name": "query_docs_filesystem_docs_by_lang_chain",
            "arguments": {"command": command},
        },
    }
    resp = requests.post(
        MCP_URL,
        json=payload,
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json, text/event-stream",
        },
        timeout=30,
    )
    return resp.text

# 示例：列出 Deep Agents 目录
result = query_docs("ls /oss/python/deepagents/")
print("=== Deep Agents 文档目录 ===")
print(result[:500])
print("...")

=== Deep Agents 文档目录 ===
event: message
data: {"result":{"content":[{"type":"text","text":"exit: 0\n--- stdout ---\nbackends.mdx\nchangelog-py.mdx\ncomparison.mdx\ncontent-builder.mdx\ncontext-engineering.mdx\ncustomization.mdx\ndata-analysis.mdx\ndeep-research.mdx\nevent-streaming.mdx\ngoing-to-production.mdx\nharness.mdx\nhuman-in-the-loop.mdx\ninterpreters.mdx\nmcp.mdx\nmemory.mdx\nmodels.mdx\noverview.mdx\npermissions.mdx\nprofiles.mdx\nquickstart.mdx\nrubric.mdx\nsandboxes.mdx\nskills.mdx\nstreaming.mdx\nsubagents.mdx\n"}]},"jsonrpc":"2.0","id":1}
...


### 4.2 读取特定文档

使用 `cat` 命令查询任何文档的原始内容：

In [ ]:
# 读取 mcp.mdx 文档
result = query_docs("cat /oss/python/deepagents/mcp.mdx")
print("=== MCP 文档内容 ===")
print(result[:2000])

=== MCP 文档内容 ===
event: message
data: {"result":{"content":[{"type":"text","text":"exit: 0\n--- stdout ---\n"}]},"jsonrpc":"2.0","id":1}


## 5. 将 MCP 集成到 Deep Agent 中

你可以在自定义 Deep Agent 中集成 MCP，让 Agent 在运行时通过 MCP 协议查询外部知识库。

In [ ]:
from langchain.tools import tool
from deepagents import create_deep_agent

@tool
def query_langchain_docs(query: str) -> str:
    """
    查询 LangChain 官方文档。
    用法：query_langchain_docs("cat /oss/python/deepagents/models.mdx")
    """
    import requests
    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "tools/call",
        "params": {
            "name": "query_docs_filesystem_docs_by_lang_chain",
            "arguments": {"command": query},
        },
    }
    resp = requests.post(
        "https://docs.langchain.com/mcp",
        json=payload,
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json, text/event-stream",
        },
        timeout=30,
    )
    return resp.text

# 创建拥有 MCP 文档查询能力的 Agent
agent = create_deep_agent(
    model="deepseek:deepseek-v4-flash",
    system_prompt="你是 LangChain 文档助手，可通过 MCP 协议实时查询官方文档。",
    tools=[query_langchain_docs],
)

print("MCP 集成 Agent 创建成功。")
print("Agent 现在可以通过 MCP 实时查询 LangChain 文档。")

MCP 集成 Agent 创建成功。
Agent 现在可以通过 MCP 实时查询 LangChain 文档。


## 6. MCP 配置方案汇总

| 客户端 | 配置文件位置 | 配置方式 |
|--------|------------|----------|
| **Claude Desktop** | `claude_desktop_config.json` | 添加 `mcpServers.langchain-docs` |
| **VSCode (Cline)** | `settings.json` | 添加 `cline.mcpServers.langchain-docs` |
| **VSCode (Continue)** | `config.json` | 添加 MCP server 到 `experimental.mcpServers` |
| **自定义 Python 应用** | 代码内配置 | HTTP POST 调用 `https://docs.langchain.com/mcp` |

## 7. 实用 MCP 查询命令

以下是一些常用的文档查询命令：

| 命令 | 用途 |
|------|------|
| `ls /oss/python/deepagents/` | 列出 Deep Agents 所有文档 |
| `cat /oss/python/deepagents/models.mdx` | 读取模型配置文档 |
| `cat /oss/python/deepagents/customization.mdx` | 读取自定义文档 |
| `cat /oss/python/deepagents/backends.mdx` | 读取后端文档 |
| `cat /oss/python/deepagents/profiles.mdx` | 读取配置档案文档 |
| `cat /oss/python/deepagents/quickstart.mdx` | 读取快速开始文档 |
| `cat /oss/python/deepagents/harness.mdx` | 读取 Harness 文档 |
| `cat /oss/python/deepagents/subagents.mdx` | 读取子 Agent 文档 |

---

**参考链接**
- [Model Context Protocol 官网](https://modelcontextprotocol.io/)
- [LangChain MCP Server](https://docs.langchain.com/mcp)
- [Claude Desktop MCP 配置](https://support.anthropic.com/en/articles/10023323)
- [MCP 规范](https://spec.modelcontextprotocol.io/)
- [Deep Agents 文档索引](https://docs.langchain.com/oss/python/deepagents/overview)